# **DEPRECATED** — use `notebooks/dermamnist_full_validation.ipynb` §4 instead.

# Final NRO Experiment — Conditional Information I(E; N | H)

**Frozen hypothesis:** Does optimization-history novelty **N** contain sample-specific information about prediction failure **E** that is not recoverable from predictive entropy **H** alone?

## Scope (strict)

- **Dataset:** DermaMNIST only
- **Checkpoints:** seeds 42, 123, 456 (frozen V5B memory — no retraining)
- **External test:** DermaMNIST-E proxy (style-shifted)
- **H:** `normalized_entropy` (= predictive entropy / log(7))
- **N:** `memory_novelty` (existing MSA implementation)
- **No** new features, optimizers, hyperparameter tuning, or test-set peeking

## Procedure

1. Fit `logit P(E|H)` and `logit P(E|H,N)` on ID **calibration** split
2. Evaluate binary log-loss on **external** test
3. Primary metric: **ΔL = L_H − L_HN** (positive ⇒ N adds information beyond H)
4. H-bin **permutation control** for N
5. Bootstrap ≥2000 replicates for 95% CIs
6. Secondary: AUROC, stratified AUROC(N|H-bin), matched high-N vs low-N pairs

Artifacts → `research/dermamnist_nro_final/`

See `.cursor/plans/final-nro-conditional-information.plan.md` for full protocol.

## Configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
import sys


def _find_repo_root(cwd: Path | None = None) -> Path:
    root = Path(cwd or Path.cwd()).resolve()
    if (root / "research").exists():
        return root
    if (root.parent / "research").exists():
        return root.parent
    return root


REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from research.common.nro_final_experiment import NROFinalConfig

cfg = NROFinalConfig(
    repo_root=REPO_ROOT,
    source_experiment_dir=REPO_ROOT / "research" / "final_experiment",
    output_dir=REPO_ROOT / "research" / "dermamnist_nro_final",
    seeds=(42, 123, 456),
    task="dermamnist",
    evaluation_domain="external",
    h_col="normalized_entropy",
    n_col="memory_novelty",
    n_bootstrap=2000,
    bootstrap_seed=20260829,
    n_h_bins=20,
    n_entropy_bins=5,
    permutation_seed=42,
    show_plots=True,
    save_artifacts=True,
)
cfg = cfg.resolve_paths()
print("Resolved configuration:")
for k, v in asdict(cfg).items():
    print(f"  {k}: {v}")

## Prerequisites

Verify frozen checkpoints and scored artifacts from `final_clinical_failure.ipynb` exist. **No retraining.**

In [ ]:
missing = []
for seed in cfg.seeds:
    ckpt = cfg.source_experiment_dir / "checkpoints" / f"{cfg.task}_seed{seed}.pkl"
    cal = cfg.source_experiment_dir / "calibration" / f"{cfg.task}_seed{seed}.csv"
    scored = cfg.source_experiment_dir / "scored" / f"{cfg.task}_seed{seed}.csv"
    for p in (ckpt, cal, scored):
        if not p.exists():
            missing.append(str(p))

if missing:
    raise FileNotFoundError(
        "Missing prerequisites. Run notebooks/final_clinical_failure.ipynb first:\n"
        + "\n".join(missing)
    )
print(f"All {len(cfg.seeds)} checkpoint + calibration + scored files found.")
print(f"Source: {cfg.source_experiment_dir}")
print(f"Output: {cfg.output_dir}")

## Run experiment

Fits logistic models on calibration, evaluates on external test, runs permutation control and bootstrap.

In [ ]:
from research.common.nro_final_experiment import run_nro_final_experiment, print_final_verdict

results = run_nro_final_experiment(cfg)
print(f"\nArtifacts written to: {results['output_dir']}")

## Results tables

In [ ]:
from IPython.display import Image, display
import pandas as pd

out = cfg.output_dir
display(pd.read_csv(out / "results.csv"))
display(pd.read_csv(out / "bootstrap_results.csv").describe())
display(pd.read_csv(out / "stratified_results.csv"))
display(pd.read_csv(out / "matched_results.csv"))

## Figures

In [ ]:
fig_dir = cfg.output_dir / "figures"
for name in [
    "failure_probability_vs_h_hn.png",
    "delta_logloss_bootstrap.png",
    "h_vs_n_scatter.png",
    "stratified_n_auroc.png",
    "matched_failure_rates.png",
]:
    path = fig_dir / name
    if path.exists():
        display(Image(filename=str(path)))

## Pre-registered PASS/FAIL verdict

In [ ]:
print_final_verdict(results)

## Interpretation

**Question:** I(E; N | H) > 0 ?

A **PASS** means:
- N reduces failure log-loss beyond H on external DermaMNIST-E (ΔL > 0, CI excludes 0)
- The gain exceeds an H-conditioned N-permutation null (ΔL_perm > 0, CI excludes 0)
- N carries **additional sample-specific failure information** given uncertainty H

A **PASS** does **not** mean:
- N fully explains or causes model failure
- N replaces predictive entropy H
- Optimization-history novelty is sufficient for deployment-ready failure detection alone

This experiment directly tests conditional information, not causal mechanism or full failure attribution.